# Predictive Modeling Using Machine Learning

Build a supervised-learning model to predict customer churn.

**Workflow:** Raw Data → Cleaning → Preprocessing → Training → Evaluation → Insights


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)


## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("customer_churn_dataset.csv")
print("Shape:", df.shape)
display(df.head())
df.info()


In [ ]:
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())
display(df.isna().sum().to_frame("Missing Values"))


## 3. Clean Duplicate Records

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", before - len(df))


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn (0 = Stayed, 1 = Churned)")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=df, x="Contract_Type", hue="Churn")
plt.title("Churn by Contract Type")
plt.xticks(rotation=15)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="Tenure_Months")
plt.title("Tenure by Churn Status")
plt.show()


## 5. Prepare Features and Preprocessing Pipeline

In [ ]:
X = df.drop(columns=["Customer_ID", "Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Train Models

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=250, max_depth=8,
        class_weight="balanced", random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
print("Models trained successfully.")


## 7. Evaluate Models

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test,pred,zero_division=0),
        "Recall": recall_score(y_test,pred,zero_division=0),
        "F1 Score": f1_score(y_test,pred,zero_division=0),
        "ROC-AUC": roc_auc_score(y_test,proba)
    }

results = pd.DataFrame([
    evaluate(logistic_model,"Logistic Regression"),
    evaluate(random_forest_model,"Random Forest")
])
display(results.round(3))


## 8. Confusion Matrix and ROC Curve

In [ ]:
best_model = random_forest_model
predictions = best_model.predict(X_test)

print(classification_report(
    y_test, predictions, target_names=["Stayed","Churned"]
))

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions,
    display_labels=["Stayed","Churned"], cmap="Blues"
)
plt.title("Confusion Matrix - Random Forest")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
RocCurveDisplay.from_estimator(logistic_model,X_test,y_test,
                               name="Logistic Regression",ax=ax)
RocCurveDisplay.from_estimator(random_forest_model,X_test,y_test,
                               name="Random Forest",ax=ax)
ax.set_title("ROC Curve Comparison")
plt.show()


## 9. Feature Importance

In [ ]:
prep = random_forest_model.named_steps["preprocessor"]
rf = random_forest_model.named_steps["model"]
names = prep.get_feature_names_out()
importance = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False).head(12)

plt.figure(figsize=(9,6))
sns.barplot(x=importance.values, y=importance.index)
plt.title("Top Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.show()
display(importance.to_frame("Importance"))


## 10. Key Insights and Conclusion

- Missing numerical values were handled using median imputation.
- Missing categorical values were handled using most-frequent imputation.
- Duplicate records were removed before modeling.
- Logistic Regression provides a linear baseline.
- Random Forest captures non-linear relationships.
- Accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, and ROC curves were used for evaluation.
- Recall is important in churn prediction because missed churners may represent lost retention opportunities.

**Correlation and model relationships should be interpreted as predictive associations, not proof of causation.**
